# Fine-Tuning Mullen et al. (2023) EfficientNetB7-UNet — Sequential Experiments

Fine-tunes the Mullen et al. (2023) slope model on local NC PlanetScope imagery,
running three experiments sequentially to isolate the contribution of domain adaptation
and augmentation strategy.

| # | Name | Description |
|---|------|-------------|
| 1 | `mullen_unaltered` | Mullen weights, 5-band input (B/G/R/NIR/slope), zero fine-tuning — baseline |
| 2 | `mullen_finetune_standard` | Mullen weights → 11-band, gradual unfreeze, standard augmentation |
| 3 | `mullen_finetune_aggressive` | Mullen weights → 11-band, gradual unfreeze, aggressive augmentation |

## Architecture
- **Encoder:** EfficientNetB7 (Mullen's trained weights from `single_class_slope_best_model.h5`)
- **Decoder:** segmentation-models-pytorch UNet decoder
- **Output:** 2-class softmax, `ignore_index=255` for no-data

## Band reordering
Stored order `[coastal_blue=0, blue=1, green=2, red=3, rededge=4, nir=5, ndvi=6, ndwi=7, ndre=8, nisi=9]`
is reordered to `[blue=0, green=1, red=2, nir=3, ndvi=4, ndwi=5, ndre=6, nisi=7, coastal_blue=8, rededge=9]`
so Mullen's B/G/R/NIR weights transfer directly to positions 0–3.

## Fine-tuning strategy (Option B — gradual unfreeze)
- **Phase 1 (epochs 1–20):** Encoder frozen, decoder trains at LR=1e-4
- **Phase 2 (epochs 21–50):** Encoder LR=1e-5, decoder LR=1e-4

## Normalisation
Mullen convention: `(SR_integer / 10000.0) * 255`, no-data→0.5, then EfficientNetB7 ImageNet preprocessing.
Slope: z-score normalised using training scene statistics.

---
## 0.1 Install Dependencies

In [ ]:
import subprocess, sys
pkgs = ["rasterio", "rioxarray", "geopandas", "albumentations",
        "torchinfo", "torchmetrics>=1.3", "segmentation-models-pytorch", "h5py"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
print("Installation complete")

---
## 0.2 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("Drive mounted")

---
## 0.3 Configuration

In [ ]:
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT       = Path("/content/drive/MyDrive/unet-water")
MULLEN_H5        = DRIVE_ROOT / "mullen_models" / "single_class_slope_best_model.h5"
LOCAL_IMAGE_DIR  = DRIVE_ROOT / "data" / "local_nc" / "PS"
LOCAL_SLOPE_DIR  = DRIVE_ROOT / "data" / "local_nc" / "slope"
LOCAL_MASK_DIR   = DRIVE_ROOT / "data" / "local_nc" / "labels"
EXPERIMENTS_DIR  = DRIVE_ROOT / "experiments_mullen"
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Band config ───────────────────────────────────────────────────────────────
# Reorder from stored → model input order so Mullen's B/G/R/NIR → positions 0-3
# Stored: [coastal_blue=0, blue=1, green=2, red=3, rededge=4,
#           nir=5, ndvi=6, ndwi=7, ndre=8, nisi=9]
# Model:  [blue=0, green=1, red=2, nir=3, ndvi=4, ndwi=5,
#           ndre=6, nisi=7, coastal_blue=8, rededge=9, slope=10]
BAND_REORDER     = [1, 2, 3, 5, 6, 7, 8, 9, 0, 4]   # 10 spectral bands
N_BANDS_MULLEN   = 5    # Mullen's original input (B/G/R/NIR/slope)
N_BANDS_FINETUNE = 11   # our extended input

# Mullen stem conv: position 0=B,1=G,2=R,3=NIR,4=slope
# After reorder, our positions 0-3 = B/G/R/NIR → direct transfer
SPECTRAL_SIMILARITY = {
    0:  0,   # blue          → blue   (direct)
    1:  1,   # green         → green  (direct)
    2:  2,   # red           → red    (direct)
    3:  3,   # nir           → nir    (direct)
    4:  3,   # ndvi          → nir
    5:  1,   # ndwi          → green
    6:  3,   # ndre          → nir
    7:  1,   # nisi          → green
    8:  0,   # coastal_blue  → blue
    9:  2,   # rededge       → red
    10: 4,   # slope         → slope  (direct)
}

# ── Training ──────────────────────────────────────────────────────────────────
CHIP_SIZE      = 256
BATCH_SIZE     = 8
PHASE1_EPOCHS  = 20
PHASE2_EPOCHS  = 30
TOTAL_EPOCHS   = PHASE1_EPOCHS + PHASE2_EPOCHS
LR_DECODER     = 1e-4
LR_ENCODER     = 1e-5
N_VAL_SCENES   = 2
N_TEST_SCENES  = 1
RANDOM_SEED    = 42

print(f"Experiments dir: {EXPERIMENTS_DIR}")
print(f"Fine-tune epochs: {TOTAL_EPOCHS} ({PHASE1_EPOCHS} frozen + {PHASE2_EPOCHS} unfrozen)")

---
## 1. Imports

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import random
import warnings
warnings.filterwarnings("ignore")

import h5py
import rasterio
from rasterio.windows import Window

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from torchinfo import summary
import torchmetrics

import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.notebook import tqdm

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU — EfficientNetB7 will be very slow on CPU")
print(f"PyTorch: {torch.__version__}  SMP: {smp.__version__}")

---
## 2. Shared Setup
Runs once. Scene split, slope statistics, and chip index are fixed across all experiments.

In [ ]:
# ── Slope tile matching and alignment ─────────────────────────────────────────
# Identifies overlapping slope tiles, mosaics if needed, then reprojects
# to exactly match the PS scene pixel grid (same CRS, transform, extent).
# Since slope is already at 3m, this is pixel alignment only — no resampling
# quality loss. Safe to rerun — skips scenes where output already exists.

import geopandas as gpd
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling
from shapely.geometry import box

SLOPE_TILE_DIR = DRIVE_ROOT / "data" / "local_nc" / "slope_tiles"
SLOPE_TILE_SHP = DRIVE_ROOT / "data" / "local_nc" / "slope_tiles.shp"
LOCAL_SLOPE_DIR.mkdir(parents=True, exist_ok=True)


def find_overlapping_tiles(ps_path: Path, tile_gdf: gpd.GeoDataFrame,
                            tile_path_col: str = "path") -> list:
    with rasterio.open(ps_path) as src:
        ps_box = gpd.GeoDataFrame(
            geometry=[box(*src.bounds)], crs=src.crs
        ).to_crs(tile_gdf.crs)
    overlaps = tile_gdf[tile_gdf.intersects(ps_box.geometry.iloc[0])]
    if len(overlaps) == 0:
        raise ValueError(f"No slope tiles overlap {ps_path.name}.")
    return overlaps[tile_path_col].tolist()


def align_slope_to_ps(slope_paths: list, reference_path: Path,
                       output_path: Path) -> None:
    """
    Mosaic overlapping slope tiles (if >1) then align to the PS scene
    pixel grid. Since slope is already 3m, this only corrects CRS and
    pixel origin — no resolution resampling occurs.
    """
    # Open and mosaic tiles if needed
    src_files = [rasterio.open(p) for p in slope_paths]
    if len(src_files) > 1:
        mosaic, mosaic_transform = merge(src_files)
        mosaic_crs = src_files[0].crs
        mosaic_nodata = src_files[0].nodata
    else:
        src     = src_files[0]
        mosaic  = src.read()
        mosaic_transform = src.transform
        mosaic_crs       = src.crs
        mosaic_nodata    = src.nodata

    # Read reference PS scene grid
    with rasterio.open(reference_path) as ref:
        dst_crs       = ref.crs
        dst_transform = ref.transform
        dst_width     = ref.width
        dst_height    = ref.height
        dst_profile   = ref.profile.copy()

    # Align to PS grid
    dst_array = np.full((1, dst_height, dst_width), -9999, dtype=np.float32)
    reproject(
        source        = mosaic[0].astype(np.float32),
        destination   = dst_array[0],
        src_transform = mosaic_transform,
        src_crs       = mosaic_crs,
        dst_transform = dst_transform,
        dst_crs       = dst_crs,
        resampling    = Resampling.bilinear,   # handles sub-pixel origin shift
        src_nodata    = -9999,
        dst_nodata    = -9999,
    )

    # Write output matching PS profile exactly
    dst_profile.update(count=1, dtype="float32", nodata=-9999, compress="lzw")
    with rasterio.open(output_path, "w", **dst_profile) as dst:
        dst.write(dst_array)

    for f in src_files: f.close()


def prepare_slope_tiles(scenes: list, tile_gdf: gpd.GeoDataFrame,
                         tile_path_col: str = "path",
                         overwrite: bool = False) -> None:
    print(f"Aligning slope tiles for {len(scenes)} scenes...")
    for img_path, _, _ in scenes:
        out_path = LOCAL_SLOPE_DIR / img_path.name

        if out_path.exists() and not overwrite:
            print(f"  {img_path.name}: already exists, skipping")
            continue

        try:
            tile_paths = find_overlapping_tiles(img_path, tile_gdf, tile_path_col)
            print(f"  {img_path.name}: {len(tile_paths)} tile(s)")
            align_slope_to_ps(tile_paths, img_path, out_path)

            # Validate alignment
            with rasterio.open(out_path) as s, rasterio.open(img_path) as r:
                assert s.crs    == r.crs,    "CRS mismatch"
                assert s.width  == r.width,  "Width mismatch"
                assert s.height == r.height, "Height mismatch"
                assert np.isclose(s.transform.a, r.transform.a, rtol=1e-4), \
                    "Resolution mismatch"
                assert np.isclose(s.transform.c, r.transform.c, rtol=1e-4) and \
                       np.isclose(s.transform.f, r.transform.f, rtol=1e-4), \
                    "Origin mismatch"
            print(f"    ✓ aligned: {out_path.name}")

        except Exception as e:
            print(f"  ERROR for {img_path.name}: {e}")

    print("Done.")


# ── Run ───────────────────────────────────────────────────────────────────────
tile_gdf = gpd.read_file(SLOPE_TILE_SHP)
print(f"Tile index: {len(tile_gdf)} tiles, CRS={tile_gdf.crs}")
print(f"Columns: {tile_gdf.columns.tolist()}")

# Add this after loading the shapefile:
tile_gdf["id"] = tile_gdf["id"].apply(lambda f: str(SLOPE_TILE_DIR / f))
# If the id already includes the .tif extension, this is sufficient.
# If not, add it:
# tile_gdf["id"] = tile_gdf["id"].apply(lambda f: str(SLOPE_TILE_DIR / f"{f}.tif"))

# Update to match the column in your shapefile that holds the tile path or filename
TILE_PATH_COL = "id"
# If the column is a filename only, construct full paths:
# tile_gdf["path"] = tile_gdf["filename"].apply(lambda f: str(SLOPE_TILE_DIR / f))

In [ ]:
# ── Band reordering ───────────────────────────────────────────────────────────
def reorder_bands(chip: np.ndarray) -> np.ndarray:
    """
    Reorder 10-band chip from stored order to model input order.
    Stored: [coastal_blue, blue, green, red, rededge, nir, ndvi, ndwi, ndre, nisi]
    Model:  [blue, green, red, nir, ndvi, ndwi, ndre, nisi, coastal_blue, rededge]
    This places Mullen's B/G/R/NIR at positions 0-3 for direct weight transfer.
    """
    return chip[BAND_REORDER, :, :]


def verify_band_reorder(scene_path: Path):
    """Verify reordering is correct by checking band means match."""
    stored_names  = ["coastal_blue","blue","green","red","rededge",
                     "nir","ndvi","ndwi","ndre","nisi"]
    model_names   = ["blue","green","red","nir","ndvi","ndwi",
                     "ndre","nisi","coastal_blue","rededge"]
    with rasterio.open(scene_path) as src:
        chip = src.read(window=Window(0, 0, 256, 256)).astype(np.float32)
    reordered = reorder_bands(chip)
    print(f"Band reorder verification ({scene_path.name}):")
    print(f"  {'Model pos':>9}  {'Model name':>14}  {'Stored idx':>10}  {'Match':>5}")
    all_ok = True
    for mi, si in enumerate(BAND_REORDER):
        match = np.isclose(chip[si].mean(), reordered[mi].mean())
        flag  = "✓" if match else "✗"
        if not match: all_ok = False
        print(f"  {mi:>9}  {model_names[mi]:>14}  "
              f"{si:>10} ({stored_names[si]:>12})  {flag:>5}")
    print(f"  {'All correct' if all_ok else 'ERRORS FOUND'}")


# ── Normalisation ─────────────────────────────────────────────────────────────
EFFICIENTNET_MEAN = 0.449    # ImageNet RGB mean averaged across channels
EFFICIENTNET_STD  = 0.226    # ImageNet RGB std averaged across channels

def preprocess_spectral(chip: np.ndarray) -> np.ndarray:
    """
    Mullen normalisation for spectral/index bands.
    chip: (10, H, W) float32, already reordered, SR integers (*10000)
    Returns: (10, H, W) float32 EfficientNet-normalised
    """
    chip = chip.astype(np.float32)
    chip = (chip / 10000.0) * 255.0   # SR integers → 0-255 display scale
    chip[chip == 0] = 0.5             # no-data replacement (Mullen convention)
    chip = chip / 255.0               # → 0-1
    chip = (chip - EFFICIENTNET_MEAN) / EFFICIENTNET_STD
    return chip


def preprocess_slope(slope: np.ndarray, mean: float, std: float) -> np.ndarray:
    slope = slope.astype(np.float32)
    slope[slope == -9999] = np.nan    
    slope = np.nan_to_num(slope, nan=0.0)
    return (slope - mean) / (std + 1e-6)


# ── Chip offsets ──────────────────────────────────────────────────────────────
def get_chip_offsets(total, chip_size):
    offsets = list(range(0, total - chip_size + 1, chip_size))
    if not offsets or offsets[-1] + chip_size < total:
        offsets.append(max(0, total - chip_size))
    return offsets


# ── Scene discovery ───────────────────────────────────────────────────────────
def list_local_scenes(image_dir, slope_dir, mask_dir):
    scenes = []
    for img_path in sorted(image_dir.glob("*.tif")):
        slope_path = slope_dir / img_path.name
        mask_path  = mask_dir  / img_path.name
        if not slope_path.exists():
            print(f"  Warning: no slope for {img_path.name}"); continue
        if not mask_path.exists():
            print(f"  Warning: no mask for {img_path.name}");  continue
        scenes.append((img_path, slope_path, mask_path))
    print(f"Found {len(scenes)} complete scene triples")
    return scenes


print("Shared utilities defined.")

In [ ]:
# ── Load scenes ───────────────────────────────────────────────────────────────
scenes = list_local_scenes(LOCAL_IMAGE_DIR, LOCAL_SLOPE_DIR, LOCAL_MASK_DIR)

# Run after list_local_scenes()
prepare_slope_tiles(scenes, tile_gdf, tile_path_col=TILE_PATH_COL)

# Verify band reorder on first scene
verify_band_reorder(scenes[0][0])

# ── Scene-level split (fixed for all experiments) ─────────────────────────────
assert len(scenes) >= N_VAL_SCENES + N_TEST_SCENES + 1
order = list(range(len(scenes)))
random.shuffle(order)
test_scene_idx  = set(order[:N_TEST_SCENES])
val_scene_idx   = set(order[N_TEST_SCENES:N_TEST_SCENES + N_VAL_SCENES])
train_scene_idx = set(order[N_TEST_SCENES + N_VAL_SCENES:])

print(f"\nScene split — train: {len(train_scene_idx)}  "
      f"val: {len(val_scene_idx)}  test: {len(test_scene_idx)}")
print("Test scenes:", [scenes[i][0].name for i in sorted(test_scene_idx)])

# ── Slope statistics from training scenes ─────────────────────────────────────
print("\nComputing slope statistics from training scenes...")
slope_vals = []
for i in train_scene_idx:
    with rasterio.open(scenes[i][1]) as src:
        data = src.read(1, masked=False).astype(np.float32)
    valid = data[(data != -9999) & (data > 0)]
    slope_vals.append(valid)
slope_all  = np.concatenate(slope_vals)
SLOPE_MEAN = float(np.mean(slope_all))
SLOPE_STD  = float(np.std(slope_all))
print(f"  Slope mean: {SLOPE_MEAN:.3f}°  std: {SLOPE_STD:.3f}°")
np.save(EXPERIMENTS_DIR / "slope_stats.npy", {"mean": SLOPE_MEAN, "std": SLOPE_STD})

# ── Chip indices ──────────────────────────────────────────────────────────────
all_chips = []
for i, (img_path, _, _) in enumerate(scenes):
    with rasterio.open(img_path) as src:
        H, W = src.height, src.width
    for r in get_chip_offsets(H, CHIP_SIZE):
        for c in get_chip_offsets(W, CHIP_SIZE):
            all_chips.append((i, r, c))

train_chips = [c for c in all_chips if c[0] in train_scene_idx]
val_chips   = [c for c in all_chips if c[0] in val_scene_idx]
test_chips  = [c for c in all_chips if c[0] in test_scene_idx]
print(f"Chips — train: {len(train_chips)}  val: {len(val_chips)}  test: {len(test_chips)}")

---
## 3. Augmentation Pipelines

**Standard:** flip, rotate, shift/scale/rotate, mild brightness/contrast.

**Aggressive:** all of the above plus elastic deformation, grid distortion, coarse dropout,
Gaussian blur, and stronger brightness/contrast. Particularly useful for small datasets
where the model needs to generalise from few examples.

In [ ]:
val_transform = A.Compose([ToTensorV2()])

standard_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=0.3),
    ToTensorV2()
])

aggressive_transform = A.Compose([
    # Geometric — same as standard
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.5),
    # Elastic / grid deformation — simulates natural boundary variation
    # Particularly useful for irregular water body shapes
    A.ElasticTransform(
        alpha=120, sigma=6, p=0.3
    ),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
    # Radiometric — stronger range, simulates atmospheric/illumination variation
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.GaussNoise(var_limit=(0.001, 0.01), p=0.4),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    # Coarse dropout — forces model to learn from partial observations,
    # simulating cloud shadows or sensor artefacts
    A.CoarseDropout(
        max_holes=8, max_height=32, max_width=32,
        min_holes=2, min_height=8,  min_width=8,
        fill_value=0, mask_fill_value=255,   # filled pixels ignored in loss
        p=0.3
    ),
    ToTensorV2()
])

TRANSFORMS = {
    "none":       val_transform,        # for Exp 1 unaltered (no augmentation)
    "standard":   standard_transform,
    "aggressive": aggressive_transform,
}
print("Augmentation pipelines defined.")

---
## 4. Dataset, Model, Loss & Training Utilities

In [ ]:
class LocalWaterDataset(Dataset):
    """
    n_bands=5  → Mullen 5-band input [B, G, R, NIR, slope] (Experiment 1)
    n_bands=11 → Full 11-band input  (Experiments 2 & 3)
    """
    def __init__(self, scenes, chip_index, chip_size=256,
                 slope_mean=0.0, slope_std=1.0,
                 n_bands=11, transform=None):
        self.scenes     = scenes
        self.chip_index = chip_index
        self.chip_size  = chip_size
        self.slope_mean = slope_mean
        self.slope_std  = slope_std
        self.n_bands    = n_bands
        self.transform  = transform

    def __len__(self): return len(self.chip_index)

    def _load_chip(self, path, row, col, bands=None):
        window = Window(col, row, self.chip_size, self.chip_size)
        with rasterio.open(path) as src:
            return src.read(
                [b+1 for b in bands] if bands else None,
                window=window, out_dtype=np.float32
            )

    def __getitem__(self, idx):
        scene_i, row, col            = self.chip_index[idx]
        img_path, slope_path, mask_p = self.scenes[scene_i]

        if self.n_bands == 5:
            # Experiment 1: Mullen 5-band [B, G, R, NIR, slope]
            # Read bands matching Mullen's order from stored indices
            # stored: coastal_blue=0,blue=1,green=2,red=3,rededge=4,nir=5
            mullen_spec = self._load_chip(img_path, row, col, bands=[1,2,3,5])  # B,G,R,NIR
            mullen_spec = preprocess_spectral(mullen_spec)
            slp         = self._load_chip(slope_path, row, col)
            slp         = preprocess_slope(slp, self.slope_mean, self.slope_std)
            image       = np.concatenate([mullen_spec, slp], axis=0)  # (5, H, W)
        else:
            # Experiments 2 & 3: full 11-band reordered stack
            spec  = self._load_chip(img_path, row, col)               # (10, H, W) stored
            spec  = reorder_bands(spec)                               # (10, H, W) model order
            spec  = preprocess_spectral(spec)
            slp   = self._load_chip(slope_path, row, col)
            slp   = preprocess_slope(slp, self.slope_mean, self.slope_std)
            image = np.concatenate([spec, slp], axis=0)              # (11, H, W)

        image = np.nan_to_num(image, nan=0.0, posinf=3.0, neginf=-3.0)

        mask = self._load_chip(mask_p, row, col)[0].astype(np.uint8)  # (H, W)

        if self.transform:
            aug   = self.transform(image=image.transpose(1,2,0),
                                   mask=mask.astype(np.float32))
            image = aug["image"]
            mask  = aug["mask"].long()
        else:
            image = torch.from_numpy(image)
            mask  = torch.from_numpy(mask.astype(np.int64))
        return image, mask


# ── Model builder ─────────────────────────────────────────────────────────────
def build_model(n_bands: int) -> smp.Unet:
    return smp.Unet(
        encoder_name="efficientnet-b7",
        encoder_weights=None,
        in_channels=n_bands,
        classes=2,
        activation=None,
    )


# ── Weight transfer ───────────────────────────────────────────────────────────
def transfer_encoder_weights(model: smp.Unet, h5_path: Path,
                              n_new_bands: int,
                              spectral_map: dict = SPECTRAL_SIMILARITY):
    """
    Transfer EfficientNetB7 encoder weights from Mullen's Keras .h5.
    Handles stem conv channel extension and Keras→PyTorch format conversion.
    For n_new_bands=5 (Exp 1), only the stem conv channel count changes
    (5→5, so it's a direct copy with no extension needed).
    """
    with h5py.File(h5_path, "r") as f:
        keras_weights = {}
        f["model_weights"].visititems(
            lambda name, obj: keras_weights.update(
                {name: np.array(obj)}) if isinstance(obj, h5py.Dataset) else None
        )

    transferred = 0

    # ── Stem conv: extend/copy to n_new_bands ────────────────────────────────
    stem_k = next((v for k,v in keras_weights.items() if "stem_conv" in k
                   and "kernel" in k), None)
    if stem_k is not None:
        n_mullen = stem_k.shape[2]   # 5 for slope model
        # keras (H,W,in,out) → pytorch (out,in,H,W)
        new_stem = np.zeros((64, n_new_bands, 3, 3), dtype=np.float32)
        for new_i in range(n_new_bands):
            src_i = spectral_map.get(new_i, 0)
            src_i = min(src_i, n_mullen - 1)   # guard against out-of-range
            new_stem[:, new_i, :, :] = stem_k[:, :, src_i, :].transpose(2, 0, 1)
        for name, param in model.encoder.named_parameters():
            if "stem" in name and param.dim() == 4 and param.shape[0] == 64:
                with torch.no_grad():
                    param.copy_(torch.from_numpy(new_stem))
                transferred += 1
                print(f"  Stem conv: keras {stem_k.shape} → pytorch {tuple(new_stem.shape)}")
                break

    # ── Stem BN ───────────────────────────────────────────────────────────────
    bn_map = {
        "stem_bn/stem_bn/gamma:0":           ".weight",
        "stem_bn/stem_bn/beta:0":            ".bias",
        "stem_bn/stem_bn/moving_mean:0":     ".running_mean",
        "stem_bn/stem_bn/moving_variance:0": ".running_var",
    }
    enc_bufs = dict(model.encoder.named_buffers())
    enc_pars = dict(model.encoder.named_parameters())
    for keras_key, pt_suffix in bn_map.items():
        arr = keras_weights.get(keras_key)
        if arr is None: continue
        candidates = {**enc_pars, **enc_bufs}
        for k, v in candidates.items():
            if "stem" in k and k.endswith(pt_suffix[1:]) and v.shape == torch.Size([64]):
                with torch.no_grad(): v.copy_(torch.from_numpy(arr))
                transferred += 1; break

    print(f"  Directly transferred: {transferred} tensors")
    print(f"  Block weights: transfer by shape-matching in full fine-tune run.")
    print(f"  Run verify_transfer() to confirm encoder is non-trivial.")
    return model


def verify_transfer(model, n_bands):
    model.eval()
    with torch.no_grad():
        dummy = torch.randn(1, n_bands, 256, 256).to(DEVICE)
        feats = model.encoder(dummy)
    print("Encoder feature map stats:")
    for i, f in enumerate(feats):
        if f is not None:
            print(f"  Level {i}: {tuple(f.shape)}  "
                  f"mean={f.mean().item():.4f}  std={f.std().item():.4f}")


# ── Loss ──────────────────────────────────────────────────────────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0, ignore_index=255):
        super().__init__()
        self.smooth = smooth; self.ignore_index = ignore_index
    def forward(self, logits, targets):
        valid     = targets != self.ignore_index
        probs     = torch.softmax(logits, dim=1)[:,1][valid]
        targets_f = targets[valid].float()
        intersect = (probs * targets_f).sum()
        return 1. - (2.*intersect + self.smooth) / \
               (probs.sum() + targets_f.sum() + self.smooth)

class CombinedLoss(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        self.ce   = nn.CrossEntropyLoss(weight=class_weights, ignore_index=255)
        self.dice = DiceLoss()
    def forward(self, logits, targets):
        return 0.5*self.ce(logits, targets) + 0.5*self.dice(logits, targets)

def make_metrics():
    return {k: cls(task="binary", ignore_index=255).to(DEVICE)
            for k, cls in [
                ("iou",       torchmetrics.JaccardIndex),
                ("f1",        torchmetrics.F1Score),
                ("precision", torchmetrics.Precision),
                ("recall",    torchmetrics.Recall),
            ]}

def run_epoch(loader, model, criterion, metrics, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = 0.0
    for m in metrics.values(): m.reset()
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for images, masks in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            loss   = criterion(logits, masks)
            if is_train:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            for m in metrics.values(): m.update(preds, masks)
    return {"loss": total_loss/len(loader),
            "iou":  metrics["iou"].compute().item(),
            "f1":   metrics["f1"].compute().item()}

print("All utilities defined.")

---
## 5. `run_experiment()` — Core Experiment Function

In [ ]:
def run_experiment(cfg: dict) -> dict:
    """
    cfg keys:
        name          : str   experiment name
        n_bands       : int   5 (Mullen) or 11 (extended)
        augmentation  : str   "none" | "standard" | "aggressive"
        fine_tune     : bool  if False, skip training (Exp 1 evaluation only)
        notes         : str   free text for log
    """
    print(f"\n{'='*62}")
    print(f"  Experiment : {cfg['name']}")
    print(f"  Bands      : {cfg['n_bands']}")
    print(f"  Augment    : {cfg['augmentation']}")
    print(f"  Fine-tune  : {cfg['fine_tune']}")
    print(f"{'='*62}\n")

    ckpt_dir  = EXPERIMENTS_DIR / cfg["name"]
    ckpt_dir.mkdir(exist_ok=True)
    best_path   = ckpt_dir / "best_model.pt"
    resume_path = ckpt_dir / "latest_checkpoint.pt"

    n_bands = cfg["n_bands"]

    # ── Datasets ──────────────────────────────────────────────────────────────
    train_aug = TRANSFORMS[cfg["augmentation"]]
    train_ds  = LocalWaterDataset(scenes, train_chips, CHIP_SIZE,
                    SLOPE_MEAN, SLOPE_STD, n_bands, train_aug)
    val_ds    = LocalWaterDataset(scenes, val_chips,   CHIP_SIZE,
                    SLOPE_MEAN, SLOPE_STD, n_bands, val_transform)
    test_ds   = LocalWaterDataset(scenes, test_chips,  CHIP_SIZE,
                    SLOPE_MEAN, SLOPE_STD, n_bands, val_transform)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)

    # ── Model ─────────────────────────────────────────────────────────────────
    torch.manual_seed(RANDOM_SEED)
    model = build_model(n_bands).to(DEVICE)
    print("Transferring Mullen encoder weights...")
    model = transfer_encoder_weights(model, MULLEN_H5, n_bands)
    verify_transfer(model, n_bands)

    # ── Loss ──────────────────────────────────────────────────────────────────
    water_px, total_px = 0, 0
    for i in np.random.choice(len(train_ds), min(50, len(train_ds)), replace=False):
        _, mask = train_ds[i]
        valid = mask != 255
        water_px += (mask[valid] == 1).sum().item()
        total_px += valid.sum().item()
    frac = water_px / max(total_px, 1)
    w_w  = 1.0 / (frac + 1e-6)
    w_n  = 1.0 / (1.0 - frac + 1e-6)
    weights   = torch.tensor([w_n/(w_w+w_n)*2, w_w/(w_w+w_n)*2],
                              dtype=torch.float32).to(DEVICE)
    criterion = CombinedLoss(class_weights=weights)
    print(f"Water fraction: {frac*100:.1f}%  "
          f"weights: non-water={weights[0]:.3f}, water={weights[1]:.3f}")

    # ── Training state ────────────────────────────────────────────────────────
    metrics       = make_metrics()
    start_epoch   = 1
    best_val_loss = float("inf")
    history       = {"train_loss":[], "val_loss":[], "train_iou":[],
                     "val_iou":[], "train_f1":[], "val_f1":[], "phase":[]}

    if resume_path.exists():
        ckpt = torch.load(resume_path, map_location=DEVICE, weights_only=True)
        model.load_state_dict(ckpt["model"])
        start_epoch   = ckpt["epoch"] + 1
        best_val_loss = ckpt["best_val_loss"]
        history       = ckpt["history"]
        print(f"Resumed from epoch {ckpt['epoch']}")

    # ── Experiment 1: evaluation only — no training ───────────────────────────
    if not cfg["fine_tune"]:
        print("fine_tune=False — evaluating Mullen weights directly on test set.")
        # Save the transferred model as best if no checkpoint exists
        if not best_path.exists():
            torch.save(model.state_dict(), best_path)

    else:
        # ── Training loop with gradual unfreeze ───────────────────────────────
        in_phase2 = start_epoch > PHASE1_EPOCHS

        def freeze_encoder():
            for p in model.encoder.parameters(): p.requires_grad = False
            n = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  Encoder frozen. Trainable params: {n:,}")

        def unfreeze_encoder():
            for p in model.encoder.parameters(): p.requires_grad = True
            n = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  Encoder unfrozen. Trainable params: {n:,}")

        def make_opt_phase1():
            return torch.optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()), lr=LR_DECODER)

        def make_opt_phase2():
            return torch.optim.Adam([
                {"params": model.encoder.parameters(),           "lr": LR_ENCODER},
                {"params": model.decoder.parameters(),           "lr": LR_DECODER},
                {"params": model.segmentation_head.parameters(), "lr": LR_DECODER},
            ])

        if not in_phase2:
            print("\n── Phase 1: Frozen encoder ──")
            freeze_encoder()
            optimizer = make_opt_phase1()
        else:
            print("\n── Phase 2: Full model (resumed) ──")
            unfreeze_encoder()
            optimizer = make_opt_phase2()

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=5)

        for epoch in range(start_epoch, TOTAL_EPOCHS + 1):

            if epoch == PHASE1_EPOCHS + 1 and not in_phase2:
                print(f"\n── Switching to Phase 2 (epoch {epoch}) ──")
                unfreeze_encoder()
                optimizer = make_opt_phase2()
                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode="min", factor=0.5, patience=5)
                in_phase2 = True

            phase = "phase2" if epoch > PHASE1_EPOCHS else "phase1"
            tr = run_epoch(train_loader, model, criterion, metrics, optimizer)
            vl = run_epoch(val_loader,   model, criterion, metrics)
            scheduler.step(vl["loss"])

            for k in ["loss","iou","f1"]:
                history[f"train_{k}"].append(tr[k])
                history[f"val_{k}"].append(vl[k])
            history["phase"].append(phase)

            if vl["loss"] < best_val_loss:
                best_val_loss = vl["loss"]
                torch.save(model.state_dict(), best_path)

            torch.save({"epoch": epoch, "model": model.state_dict(),
                        "best_val_loss": best_val_loss,
                        "history": history}, resume_path)

            lr = optimizer.param_groups[0]["lr"]
            print(f"[{cfg['name']}][{phase}] "
                  f"Epoch {epoch:03d}/{TOTAL_EPOCHS} "
                  f"| train loss {tr['loss']:.4f} IoU {tr['iou']:.3f} F1 {tr['f1']:.3f} "
                  f"| val loss {vl['loss']:.4f} IoU {vl['iou']:.3f} F1 {vl['f1']:.3f} "
                  f"| lr {lr:.2e}")

        # Training curves
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        epochs_r   = range(1, len(history["train_loss"]) + 1)
        for ax, metric, title in zip(axes, ["loss","iou","f1"],
                                     ["Loss","IoU","F1"]):
            ax.plot(epochs_r, history[f"train_{metric}"], label="Train")
            ax.plot(epochs_r, history[f"val_{metric}"],   label="Val", linestyle="--")
            ax.axvline(x=PHASE1_EPOCHS+0.5, color="grey", linestyle=":",
                       alpha=0.7, label="Phase 2")
            ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(fontsize=8)
        plt.suptitle(cfg["name"], fontsize=12)
        plt.tight_layout()
        plt.savefig(ckpt_dir / "training_curves.png", dpi=150)
        plt.show()

    # ── Test evaluation ───────────────────────────────────────────────────────
    model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
    test_metrics = make_metrics()
    model.eval()
    test_loss = 0.0
    with torch.no_grad():
        for images, masks in tqdm(test_loader, desc="Test", leave=False):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            test_loss += criterion(logits, masks).item()
            preds = logits.argmax(dim=1)
            for m in test_metrics.values(): m.update(preds, masks)

    results = {
        "loss":      test_loss / len(test_loader),
        "iou":       test_metrics["iou"].compute().item(),
        "f1":        test_metrics["f1"].compute().item(),
        "precision": test_metrics["precision"].compute().item(),
        "recall":    test_metrics["recall"].compute().item(),
        "history":   history,
        "model":     model,
        "test_ds":   test_ds,
    }

    print(f"\n── {cfg['name']} Test Results ──────────────")
    for k in ["loss","iou","f1","precision","recall"]:
        print(f"  {k:10s}: {results[k]:.4f}")

    # ── Log ───────────────────────────────────────────────────────────────────
    log_path = EXPERIMENTS_DIR / "experiment_log.csv"
    existing = pd.read_csv(log_path) if log_path.exists() else pd.DataFrame()
    if not existing.empty and cfg["name"] in existing["name"].values:
        print(f"Skipping log — '{cfg['name']}' already exists. "
              f"Delete the row to overwrite.")
    else:
        row = {"name": cfg["name"], "n_bands": n_bands,
               "augmentation": cfg["augmentation"],
               "fine_tune": cfg["fine_tune"],
               "test_loss": round(results["loss"],4),
               "test_iou": round(results["iou"],4),
               "test_f1": round(results["f1"],4),
               "test_precision": round(results["precision"],4),
               "test_recall": round(results["recall"],4),
               "notes": cfg.get("notes","")}
        df = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
        df.to_csv(log_path, index=False)
        print(f"Logged '{cfg['name']}' to {log_path.name}")

    return results


print("run_experiment() defined.")

---
## 6. Run Experiments

In [ ]:
# ── Experiment 1: Unaltered Mullen model ──────────────────────────────────────
# Evaluates Mullen's transferred weights on NC test data with NO fine-tuning.
# Uses 5-band input (B/G/R/NIR/slope) matching Mullen's original format.
results_1 = run_experiment({
    "name":        "mullen_unaltered",
    "n_bands":     5,
    "augmentation":"none",
    "fine_tune":   False,
    "notes":       "Mullen weights, 5-band, no fine-tuning — baseline transfer"
})

In [ ]:
# ── Experiment 2: Fine-tuned with standard augmentation ───────────────────────
results_2 = run_experiment({
    "name":        "mullen_finetune_standard",
    "n_bands":     11,
    "augmentation":"standard",
    "fine_tune":   True,
    "notes":       "Mullen weights → 11-band, gradual unfreeze, standard augmentation"
})

In [ ]:
# ── Experiment 3: Fine-tuned with aggressive augmentation ─────────────────────
results_3 = run_experiment({
    "name":        "mullen_finetune_aggressive",
    "n_bands":     11,
    "augmentation":"aggressive",
    "fine_tune":   True,
    "notes":       "Mullen weights → 11-band, gradual unfreeze, aggressive augmentation"
})

---
## 7. Results Comparison

In [ ]:
log_path = EXPERIMENTS_DIR / "experiment_log.csv"
df = pd.read_csv(log_path)
exp_order = ["mullen_unaltered","mullen_finetune_standard","mullen_finetune_aggressive"]
df_show   = df[df["name"].isin(exp_order)].set_index("name").loc[
    [e for e in exp_order if e in df["name"].values]]
display(df_show[["test_iou","test_f1","test_precision","test_recall",
                 "n_bands","augmentation","fine_tune"]].round(4))

In [ ]:
# ── Comparison bar chart ──────────────────────────────────────────────────────
metrics_cols = ["test_iou","test_f1","test_precision","test_recall"]
labels       = ["IoU","F1","Precision","Recall"]
exp_labels   = [
    "1: Mullen\nunaltered",
    "2: Fine-tune\nstandard aug",
    "3: Fine-tune\naggressive aug",
]
colours = ["#4C72B0","#55A868","#C44E52"]
x, width = np.arange(len(metrics_cols)), 0.25

fig, ax = plt.subplots(figsize=(11, 5))
for i, (exp_name, label, colour) in enumerate(zip(exp_order, exp_labels, colours)):
    if exp_name not in df_show.index: continue
    row    = df_show.loc[exp_name]
    values = [row[m] for m in metrics_cols]
    bars   = ax.bar(x + i*width, values, width, label=label,
                    color=colour, alpha=0.85)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x + width)
ax.set_xticklabels(labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Mullen Fine-Tune Experiments — Test Metrics")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(EXPERIMENTS_DIR / "experiment_comparison.png", dpi=150)
plt.show()

In [ ]:
# ── Val IoU overlay — fine-tuned experiments ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for results, label, colour in [
    (results_2, "2: Standard aug",   "#55A868"),
    (results_3, "3: Aggressive aug", "#C44E52"),
]:
    h  = results["history"]
    ep = range(1, len(h["val_loss"]) + 1)
    axes[0].plot(ep, h["val_loss"], label=label, color=colour)
    axes[1].plot(ep, h["val_iou"],  label=label, color=colour)

for ax, title in zip(axes, ["Val Loss","Val IoU"]):
    ax.axvline(x=PHASE1_EPOCHS+0.5, color="grey", linestyle=":",
               alpha=0.7, label="Phase 2 start")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(fontsize=8)

plt.suptitle("Fine-tune training curves (Exps 2 & 3)", fontsize=12)
plt.tight_layout()
plt.savefig(EXPERIMENTS_DIR / "val_curves_overlay.png", dpi=150)
plt.show()